<a href="https://colab.research.google.com/github/ounissinour/PolypVision/blob/milestone-2-final-test/notebooks/03b_PolypPVT_Final_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 2 – final test of the validation-selected Polyp-PVT

**PolypVision** · Nour Ounissi (RSPN69), Taha Mersni (K1TO74), Baha Amar (ZXQA2V)

`03_PolypPVT_Retraining.ipynb` trained six configurations and **locked** the winner on validation only:
`size_288` (structure loss, no augmentation, 288 × 288), epoch 100, validation mean Dice 0.9196 / IoU 0.8678
(`results/validation_selection.json`). Its test evaluation never produced numbers: the overlap guard stopped
it (49 test images look like training images) and the last attempt crashed with `NameError: json`.

This notebook finishes Milestone 2: it scores **that one locked checkpoint once** on the five official test sets.
Nothing is re-selected or re-tuned here.

**About the 49 flagged images.** Most are CVC-ClinicDB test frames whose neighbouring video frames are in the
training set. This is a property of the *official benchmark split* that both papers and every comparison table
use (the authors' 0.937 on ClinicDB is measured the same way), not a leak created by our pipeline. So we report
two numbers per test set:

* **full** – all images, directly comparable with the papers and with Milestone 1;
* **excluding flagged** – without the 49 near-duplicate candidates, to show how much the overlap matters.

| Step | What happens |
|---|---|
| 1 | Settings, packages, official Polyp-PVT code (pinned), Baha's `m2_pipeline.py` and result files from the repository |
| 2 | Data: test sets (and the training set, only if the checkpoint has to be re-created) |
| 3 | The locked checkpoint: load it from Drive, check it **is** `size_288` and re-check its validation score; if it is not on this Drive, optionally re-train it with the identical recipe |
| 4 | One test evaluation: per-image and per-dataset mean Dice / IoU (256 thresholds, as the papers and Milestone 1), full and excluding flagged |
| 5 | Comparison with the authors' Polyp-PVT and PraNet (Milestone 1 per-image scores, same metric) |
| 6 | Figures and saving the results |

## 1. Settings

In [ ]:
import os
BRANCH = "milestone-2-final-test"   # repository branch that holds m2_pipeline.py and notebooks/results
REPO_RAW = f"https://raw.githubusercontent.com/ounissinour/PolypVision/{BRANCH}"
PVT_COMMIT = "cd71fbf9fcd8c32cc0f33699f4e0186073c64dc4"

# The checkpoint selected in 03_PolypPVT_Retraining.ipynb. It lives in the Drive of whoever ran that notebook
# (MyDrive/PolypVision_M2/runs/size_288/best.pt). If Baha shared the folder with you, add a shortcut to
# "MyDrive" (right click > Organize > Add shortcut) and this path works unchanged.
CHECKPOINT = "/content/drive/MyDrive/PolypVision_M2/runs/size_288/best.pt"
RETRAIN_IF_MISSING = True   # no checkpoint found -> re-train size_288 with the identical recipe (~1.5 h on a T4)

from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/PolypVision_M2"          # same folder as 03_PolypPVT_Retraining.ipynb
DATA = "/content/polyp_data"
OUT = f"{WORK}/final_test"
for d in [WORK, f"{WORK}/cache", f"{WORK}/runs", DATA, OUT]:
    os.makedirs(d, exist_ok=True)
os.environ.update(WORK=WORK, DATA=DATA, REPO_RAW=REPO_RAW, PVT_COMMIT=PVT_COMMIT)
print("checkpoint:", CHECKPOINT, "| exists:", os.path.exists(CHECKPOINT))

In [ ]:
%pip -q install gdown pysodmetrics timm tabulate

In [ ]:
%%bash
set -e
cd /content
[ -d Polyp-PVT ] || git clone -q --filter=blob:none https://github.com/DengPingFan/Polyp-PVT
git -C Polyp-PVT checkout -q --detach $PVT_COMMIT && git -C Polyp-PVT log -1 --format='Polyp-PVT %h %cd'
mkdir -p /content/repo
for f in src/training/m2_pipeline.py notebooks/results/validation_selection.json notebooks/results/validation_summary.csv \
         notebooks/results/training_split_manifest.csv notebooks/results/overlap_audit_candidates.csv \
         milestone1/results/PraNet_per_image.csv milestone1/results/Polyp-PVT_per_image.csv; do
  wget -q -O /content/repo/$(basename $f) $REPO_RAW/$f || { echo "missing $f on branch" >&2; exit 1; }
done
cp /content/repo/m2_pipeline.py /content/m2_pipeline.py
ls -l /content/repo

## 2. Data and backbone

Same Google Drive links as Milestone 1 and `03_PolypPVT_Retraining.ipynb`. The PVTv2-B2 backbone file is needed
because the `PolypPVT` constructor loads it; its weights are then replaced by the checkpoint.

In [ ]:
%%bash
set -euo pipefail
cd $WORK/cache
[ -f TestDataset.zip ] || gdown -q 1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao -O TestDataset.zip
if [ ! -f pvt_v2_b2.pth ]; then
  gdown -q --folder 1Eu8v9vMRvt-dyCH0XSV2i77lAd62nPXV -O backbone_download
  find backbone_download -name 'pvt_v2_b2*.pth' -exec cp {} pvt_v2_b2.pth \; -quit
  rm -rf backbone_download
fi
rm -rf $DATA/TestDataset && unzip -q -o TestDataset.zip -d $DATA && rm -rf $DATA/__MACOSX
mkdir -p /content/Polyp-PVT/pretrained_pth && cp -n pvt_v2_b2.pth /content/Polyp-PVT/pretrained_pth/
for d in Kvasir:100 CVC-ClinicDB:62 CVC-ColonDB:380 ETIS-LaribPolypDB:196 CVC-300:60; do
  n=$(ls $DATA/TestDataset/${d%%:*}/images | wc -l); printf '%-18s %4s images (expected %s)\n' ${d%%:*} $n ${d##*:}
  [ $n -eq ${d##*:} ] || { echo "test set incomplete" >&2; exit 1; }
done

## 3. The locked checkpoint

Loaded from Drive and checked: its config must be exactly the locked `size_288` run, and re-scoring the 90
validation images must reproduce the locked validation Dice (0.9196). If the file is not on this Drive and
`RETRAIN_IF_MISSING` is on, `size_288` is re-trained here with Baha's `train_experiment` and the identical
settings (same split manifest, seed 2026, 100 epochs, batch 8, AdamW 1e-4, AMP). GPU training is not bit-for-bit
deterministic, so a re-trained checkpoint is a faithful **re-run**, not the identical file; the notebook says
which of the two was used.

In [ ]:
import json, sys, hashlib
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch
import torch.nn.functional as F
sys.path.insert(0, "/content")
import m2_pipeline as m2

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda")
lock = json.loads(Path("/content/repo/validation_selection.json").read_text())
LOCKED = {"name": "size_288", "size": 288, "loss": "structure", "augmentation": False}
assert lock["selected_experiment"] == LOCKED["name"], lock
print(json.dumps(lock, indent=2))

def make_model():
    os.chdir("/content/Polyp-PVT")
    from lib.pvt import PolypPVT
    return PolypPVT()

manifest = pd.read_csv("/content/repo/training_split_manifest.csv")
print(manifest.groupby(["split", "source"]).size().unstack(fill_value=0))

# training images: needed to re-check the validation score (and to re-train if necessary)
train_zip = Path(WORK) / "cache" / "TrainDataset.zip"
if not train_zip.exists():
    os.system(f"gdown -q 1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb -O '{train_zip}'")
if not Path(DATA, "TrainDataset", "image").is_dir():
    os.system(f"unzip -q -o '{train_zip}' -d {DATA} && rm -rf {DATA}/__MACOSX")

checkpoint_source = "locked checkpoint from 03_PolypPVT_Retraining.ipynb"
if not Path(CHECKPOINT).exists():
    if not RETRAIN_IF_MISSING:
        raise FileNotFoundError(f"{CHECKPOINT} not found. Share/shortcut Baha's PolypVision_M2 folder, or set RETRAIN_IF_MISSING = True.")
    checkpoint_source = "re-trained here with the identical size_288 recipe (checkpoint not available on this Drive)"
    runtime = {"torch": torch.__version__, "gpu": torch.cuda.get_device_name(0)}
    m2.train_experiment(LOCKED, make_model, manifest[manifest.split == "train"].reset_index(drop=True),
                        manifest[manifest.split == "val"].reset_index(drop=True), DATA, f"{WORK}/runs", runtime,
                        seed=2026, epochs=100, batch_size=8, learning_rate=1e-4, weight_decay=1e-4,
                        grad_clip=0.5, amp=True, resume=True)
    CHECKPOINT = f"{WORK}/runs/size_288/best.pt"
print("checkpoint source:", checkpoint_source)

In [ ]:
ckpt = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
assert ckpt["config"] == LOCKED, f"not the locked run: {ckpt['config']}"
sha = hashlib.sha256(Path(CHECKPOINT).read_bytes()).hexdigest()
model = make_model().to(DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"config {ckpt['config']} | epoch {ckpt['epoch']} | stored val Dice {ckpt['val_mean_dice']:.6f} | sha256 {sha[:16]}…")

val_rows = manifest[manifest.split == "val"].reset_index(drop=True)
if Path(DATA, "TrainDataset", "image").is_dir():
    val_dice, val_iou = m2.validate(model, val_rows, DATA, LOCKED["size"], DEVICE)
    print(f"re-scored validation: Dice {val_dice:.6f} / IoU {val_iou:.6f}  (stored {ckpt['val_mean_dice']:.6f} / {ckpt['val_mean_iou']:.6f})")
    assert abs(val_dice - ckpt["val_mean_dice"]) < 1e-4, "checkpoint does not reproduce its validation score"
else:
    print("TrainDataset not unpacked: validation re-check skipped (the stored score is printed above)")
if checkpoint_source.startswith("locked"):
    assert abs(ckpt["val_mean_dice"] - lock["validation_mean_dice"]) < 1e-6, "this is not the locked checkpoint"

## 4. One test evaluation

Prediction exactly as in Baha's `evaluate_locked_test` (and validation): resize to 288 with OpenCV, ImageNet
normalisation, P1 + P2, bilinear upsampling to the mask size, sigmoid, ×255. Scoring exactly as Milestone 1:
pysodmetrics `FmeasureV2`, 256 thresholds, sample-based mean Dice / IoU (Dice@0.5 for reference). Scores are kept
per image, so the "excluding flagged" numbers are simply the mean over the remaining images (the 256-threshold
mean is linear in the images, so this equals scoring the reduced set).

In [ ]:
from py_sod_metrics import DICEHandler, FmeasureV2, IOUHandler
TEST = ["Kvasir", "CVC-ClinicDB", "CVC-ColonDB", "ETIS-LaribPolypDB", "CVC-300"]
flagged = pd.read_csv("/content/repo/overlap_audit_candidates.csv")
flagged_keys = set(zip(flagged.dataset, flagged.test_stem.astype(str)))
print("flagged per dataset:", flagged.dataset.value_counts().to_dict())

@torch.no_grad()
def predict(model, image_bgr, size, out_hw):
    image = cv2.resize(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB), (size, size), interpolation=cv2.INTER_LINEAR)
    x = (image.astype(np.float32) / 255.0 - m2.IMAGENET_MEAN) / m2.IMAGENET_STD
    p1, p2 = model(torch.from_numpy(np.ascontiguousarray(x)).permute(2, 0, 1)[None].to(DEVICE))
    prob = torch.sigmoid(F.interpolate(p1 + p2, size=out_hw, mode="bilinear", align_corners=False))[0, 0]
    return np.clip(prob.float().cpu().numpy() * 255, 0, 255).astype(np.uint8)

rows = []
for dataset in TEST:
    root = Path(DATA, "TestDataset", dataset)
    images = m2._index_by_stem(root / "images", m2.IMAGE_EXTENSIONS)
    masks = m2._index_by_stem(root / "masks", m2.MASK_EXTENSIONS)
    assert set(images) == set(masks) and len(images) == m2.TEST_COUNTS[dataset]
    dice = DICEHandler(with_dynamic=True, with_adaptive=False, with_binary=True, sample_based=True)
    iou = IOUHandler(with_dynamic=True, with_adaptive=False, with_binary=True, sample_based=True)
    metric = FmeasureV2(metric_handlers={"dice": dice, "iou": iou})
    stems = sorted(images)
    for stem in stems:
        gt = cv2.imread(str(masks[stem]), cv2.IMREAD_GRAYSCALE)
        pred = predict(model, cv2.imread(str(images[stem])), LOCKED["size"], gt.shape)
        metric.step(pred=pred, gt=((gt > 127) * 255).astype(np.uint8), normalize=True)
    rows.append(pd.DataFrame({
        "dataset": dataset, "stem": stems,
        "dice": np.mean(dice.dynamic_results, axis=1), "iou": np.mean(iou.dynamic_results, axis=1),
        "dice_0.5": np.ravel(dice.binary_results), "iou_0.5": np.ravel(iou.binary_results),
    }))
    print(f"{dataset:18s} n={len(stems):4d} meanDice={rows[-1].dice.mean():.4f}")
per_image = pd.concat(rows, ignore_index=True)
per_image["flagged"] = [(d, str(s)) in flagged_keys for d, s in zip(per_image.dataset, per_image.stem)]
assert per_image.flagged.sum() == len(flagged), "flagged list does not match the test file names"
per_image.to_csv(f"{OUT}/size_288_test_per_image.csv", index=False)

def summarise(df, label):
    out = []
    for dataset in TEST:
        g = df[df.dataset == dataset]
        clean = g[~g.flagged]
        out.append({"model": label, "dataset": dataset, "n": len(g), "meanDice": g.dice.mean(), "meanIoU": g.iou.mean(),
                    "Dice@0.5": g["dice_0.5"].mean(), "n_excl_flagged": len(clean),
                    "meanDice_excl_flagged": clean.dice.mean(), "meanIoU_excl_flagged": clean.iou.mean()})
    return pd.DataFrame(out)

ours = summarise(per_image, "Polyp-PVT size_288, ours (validation-selected)")
ours.to_csv(f"{OUT}/final_test_metrics.csv", index=False)
print(ours.round(4).to_string(index=False))

## 5. Comparison with the authors' Polyp-PVT and PraNet

Milestone 1 stored per-image scores of the authors' released models, computed with the same metric
(`milestone1/results/*_per_image.csv`), so the same "excluding flagged" subsets can be scored for them without
re-running anything. Reminder: the authors' Polyp-PVT checkpoint was chosen **on the test sets** by the official
`Train.py`; ours was chosen on validation only.

In [ ]:
def from_m1(path, label):
    df = pd.read_csv(path)
    df["stem"] = df.image.str.rsplit(".", n=1).str[0]
    df = df.rename(columns={"dice_mean_thr": "dice", "iou_mean_thr": "iou"})
    df["flagged"] = [(d, str(s)) in flagged_keys for d, s in zip(df.dataset, df.stem)]
    assert df.flagged.sum() == len(flagged), f"{label}: flagged images not all found"
    return summarise(df, label)

authors = from_m1("/content/repo/Polyp-PVT_per_image.csv", "Polyp-PVT, authors (epoch chosen on test)")
pranet = from_m1("/content/repo/PraNet_per_image.csv", "PraNet, authors")
assert abs(authors.set_index("dataset").loc["CVC-ColonDB", "meanDice"] - 0.808) < 0.001
allres = pd.concat([pranet, authors, ours], ignore_index=True)
allres.to_csv(f"{OUT}/final_comparison_long.csv", index=False)

def wide(col):
    t = allres.pivot(index="model", columns="dataset", values=col)[TEST]
    t["mean of 5"] = t.mean(axis=1)
    return t.loc[[pranet.model[0], authors.model[0], ours.model[0]]]

full, clean = wide("meanDice"), wide("meanDice_excl_flagged")
full.to_csv(f"{OUT}/final_comparison_full.csv"); clean.to_csv(f"{OUT}/final_comparison_excl_flagged.csv")
with open(f"{OUT}/final_comparison.md", "w") as f:
    f.write("Mean Dice, all test images\n\n" + full.round(3).to_markdown() +
            "\n\nMean Dice, excluding the 49 flagged near-duplicate candidates\n\n" + clean.round(3).to_markdown() +
            "\n\nMean IoU, all test images\n\n" + wide("meanIoU").round(3).to_markdown())
print("Mean Dice, all test images\n", full.round(3).to_string(), "\n")
print("Mean Dice, excluding flagged\n", clean.round(3).to_string(), "\n")
print("Mean IoU, all test images\n", wide("meanIoU").round(3).to_string())

## 6. Figures and record

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), sharey=True)
for ax, (t, title) in zip(axes, [(full, "all test images"), (clean, "excluding the 49 flagged images")]):
    t[TEST].T.plot.bar(ax=ax, width=0.8, rot=0)
    ax.set_title(f"mean Dice – {title}"); ax.set_ylim(0.5, 1.0); ax.legend(fontsize=7, loc="lower left")
plt.tight_layout(); plt.savefig(f"{OUT}/final_comparison.png", dpi=150); plt.show()

# hardest and easiest test images for our model (ColonDB and ETIS, the unseen sets)
show = pd.concat([per_image[per_image.dataset == d].nsmallest(2, "dice") for d in ["CVC-ColonDB", "ETIS-LaribPolypDB"]] +
                 [per_image[per_image.dataset == d].nlargest(1, "dice") for d in ["CVC-ColonDB", "ETIS-LaribPolypDB"]])
fig, axes = plt.subplots(len(show), 3, figsize=(9, 2.8 * len(show)))
for ax_row, r in zip(axes, show.itertuples()):
    root = Path(DATA, "TestDataset", r.dataset)
    img = cv2.imread(str(m2._index_by_stem(root / "images", m2.IMAGE_EXTENSIONS)[r.stem]))
    gt = cv2.imread(str(m2._index_by_stem(root / "masks", m2.MASK_EXTENSIONS)[r.stem]), 0)
    pred = predict(model, img, LOCKED["size"], gt.shape)
    for ax, (im, title) in zip(ax_row, [(img[:, :, ::-1], f"{r.dataset}/{r.stem}"), (gt, "ground truth"), (pred, f"ours, Dice {r.dice:.2f}")]):
        ax.imshow(im, cmap=None if im.ndim == 3 else "gray"); ax.set_title(title, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/examples.png", dpi=110); plt.show()

record = {"selected_experiment": "size_288", "checkpoint": CHECKPOINT, "checkpoint_sha256": sha,
          "checkpoint_source": checkpoint_source, "checkpoint_epoch": int(ckpt["epoch"]),
          "validation_mean_dice": float(ckpt["val_mean_dice"]), "test_evaluated_once": True,
          "test_evaluated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
          "flagged_images_excluded_in_second_table": int(len(flagged)),
          "files": ["final_test_metrics.csv", "size_288_test_per_image.csv", "final_comparison_full.csv",
                    "final_comparison_excl_flagged.csv", "final_comparison.md", "final_comparison.png", "examples.png"]}
Path(OUT, "final_test_record.json").write_text(json.dumps(record, indent=2))
print(json.dumps(record, indent=2))
print("\nCopy the folder", OUT, "into notebooks/results/final_test/ in the repository.")